**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 05: Advanced RAG - HyDE, Reranking, Ensemble

## 이 세션에서 할 일

기본 RAG에 기법 네 개를 얹어보고, **정말 좋아지는지 숫자로 확인**합니다.

| 기법 | 한 줄 설명 |
|------|-----------|
| **HyDE** | 질문으로 가짜 답변을 만들고, 그걸로 검색 |
| **Reranking** | 넓게 찾은 뒤, 더 똑똑한 모델로 순서를 다시 매김 |
| **Ensemble** | 키워드 검색(BM25) + 의미 검색을 합침 |
| **Parent Document** | 작은 조각으로 찾고, 큰 조각을 돌려줌 |

## 미리 말해두는 결론

**기법마다 이득이 천차만별이고, 설정을 틀리면 이득이 통째로 사라집니다.**
실제로 측정한 결과입니다. 기준선인 기본 RAG는 **Hit@1 60% / Hit@3 83% / 12ms** 입니다.

| 기법 | Hit@1 | Hit@3 | 검색 시간 | 한 줄 평 |
|------|-------|-------|-----------|----------|
| Reranking | **93%** | **97%** | 69ms | 압도적. 단, 다국어 모델을 써야 함 |
| Parent Document | 77% | 90% | 11ms | 좋아지는데 느려지지도 않음 |
| Ensemble | 70% | 83% | 10ms | 좋아짐. 가중치를 잘못 주면 이득이 사라짐 |
| HyDE | 60~67% | 90~93% | **1,400ms** | 조금 오르지만 **100배 느림** |

그리고 기법보다 **모델을 제대로 고르는 게 훨씬 중요합니다.**
한국어 문서에 영어 임베딩 모델을 쓰면 Hit@1이 60%에서 **17%** 로 무너집니다.
그 상태에서는 무슨 기법을 얹어도 소용없습니다.

마지막 9️⃣에서는 이 비교를 **Streamlit 웹 앱**으로 만들어,
아무 질문이나 던지고 방법별 결과를 나란히 눈으로 확인합니다.

---

### 실습 환경
- **GPU**: 없어도 됨 (있으면 Reranking이 빠름)
- **패키지**: langchain, chromadb, sentence-transformers, rank_bm25, langchain-openai
- **모델 다운로드**: 최초 1회 약 2.7GB (Reranking 모델이 2.2GB)
- **HyDE 부분만 OpenAI API 키 필요** (없으면 그 셀만 자동으로 건너뜁니다)


In [ ]:
# 📦 패키지 확인
import importlib
import os

packages = [
    "langchain",
    "langchain_community",
    "chromadb",
    "sentence_transformers",
    "rank_bm25",
    "langchain_openai",   # HyDE 섹션에서만 사용
]

print("📦 패키지 버전 확인")
print("=" * 40)
for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치 필요")
        print(f"     pip install {pkg_name.replace('_', '-')}")

In [ ]:
# 🔧 GPU 메모리 유틸리티 (GPU 사용 시)
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print(f"[{tag}] CPU 모드로 실행 중")

# 리랭커는 GPU에서 훨씬 빠르다
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print_gpu_memory("초기 상태")
print(f"사용할 디바이스: {DEVICE}")

---

## 1️⃣ 기본 RAG는 무엇이 부족한가

기본 RAG는 "질문 벡터와 가까운 문서"를 찾는 게 전부입니다. 그래서 이런 데서 실패합니다.

| 실패하는 상황 | 이걸 노리는 기법 |
|---------------|------------------|
| 질문은 짧은데 문서는 길어서 모양이 안 맞음 | HyDE |
| 찾긴 찾았는데 엉뚱한 게 1등 | Reranking |
| "GRPO" 같은 약어를 못 알아봄 | Ensemble (BM25) |
| 조각이 너무 작아 앞뒤 맥락이 잘림 | Parent Document |

> 이 표는 아직 **주장**입니다. 정말 그런지는 재봐야 압니다.
> 재려면 먼저 **"좋아졌다"를 어떻게 판정할지** 정해야 합니다. 그게 2️⃣입니다.


---

## 2️⃣ 채점 방법 정하기

### 흔한 실수: 검색기한테 자기 시험지를 채점하게 시키기

이런 코드를 자주 봅니다.

```python
# ❌ 이러면 안 된다
점수 = 질문과 검색된 문서의 코사인 유사도 평균
```

문제는 **기본 검색기가 하는 일이 정확히 이 점수를 최대로 만드는 것**이라는 겁니다.
그러니 기본 RAG가 항상 1등입니다. 다른 기법은 이길 방법이 없습니다.
**시험 문제를 출제자가 낸 셈**이라 결과가 이미 정해져 있습니다.

### 그래서: 사람이 정답을 미리 정해둡니다

질문마다 **"이 문서에 답이 있다"** 를 손으로 적어둡니다. 그리고 검색 결과와 대조합니다.

```
질문: "GRPO가 뭐야?"        정답 문서: rl_methods.txt

검색 1등 → rl_methods.txt   ✅ 맞음
검색 2등 → alignment.txt
검색 3등 → llm_basics.txt
```

### 점수는 두 개, 그리고 시간

| | 뜻 |
|---|-----|
| **Hit@1** | 1등으로 가져온 문서가 정답인 질문의 비율 |
| **Hit@3** | 정답이 3등 안에 든 질문의 비율 |
| **검색 시간** | 질문 1개를 처리하는 데 걸린 평균 시간 |

**RAG에서는 Hit@3이 제일 중요합니다.** 검색된 3개가 전부 LLM한테 넘어가므로,
1등이 아니어도 3등 안에만 있으면 답을 만들 재료는 확보한 셈입니다.

**Hit@1은 순서가 얼마나 정확한지**를 봅니다.
Hit@3은 그대로인데 Hit@1만 올랐다면, 새로 찾아낸 게 아니라
**원래 3등 안에 있던 걸 1등으로 끌어올린 것**입니다.

검색 시간은 같이 봐야 합니다. 점수가 조금 오르는데 100배 느려지면 쓸 수 없습니다.

### 시험지 구성

- **문서 24개** (LLM, 파인튜닝, 벡터DB, 프롬프트, 강화학습, RAG, 에이전트 …)
- **질문 30개**, 정답 문서를 사람이 직접 표시


> 💡 문서를 5개만 쓰면 3개 뽑기가 전체의 30%라 기본 RAG가 이미 만점입니다.
> 그러면 무슨 기법을 써도 좋아질 자리가 없습니다. 시험지를 적당히 어렵게 만드는 것도 준비의 일부입니다.


In [ ]:
# 📄 시험지 — 문서 24개와 정답표
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- 코퍼스: 주제가 서로 겹치도록 24개 문서 구성 ---
# (source, topic, content) — 주제가 인접해야 "헷갈릴 여지"가 생기고, 그래야 기법 차이가 드러난다
RAW_DOCS = [
("llm_basics.txt", "LLM", """대규모 언어 모델(LLM)은 수십억 개의 파라미터를 가진 딥러닝 모델입니다.
GPT-4, Claude, Gemini 등이 대표적인 LLM이며, 대량의 텍스트 코퍼스로 사전학습됩니다.
사전학습 단계에서는 다음 토큰을 예측하는 단순한 목표로 언어의 통계적 구조를 습득합니다.
이후 지시 조정(instruction tuning)을 거쳐 사람의 지시를 따르는 형태로 다듬어집니다.
LLM의 핵심 능력은 긴 문맥을 이해하고 자연스러운 텍스트를 생성하는 것입니다.
파라미터 수가 커질수록 성능이 향상되는 경향을 스케일링 법칙이라 부릅니다.
다만 최근에는 무조건 크기를 키우기보다 데이터 품질을 높이는 방향이 중시됩니다."""),
("slm.txt", "LLM", """소형 언어 모델(sLLM)은 1B에서 7B 규모의 경량 언어 모델을 가리킵니다.
Phi-3, Gemma, Qwen2.5 등이 대표적이며 온디바이스 추론과 저비용 서빙에 적합합니다.
큰 모델 대비 성능 손실이 크지 않아 특정 도메인에서는 충분한 대안이 됩니다.
소형 모델은 응답 지연이 짧고 GPU 메모리를 적게 써서 동시 처리량이 높습니다.
개인정보를 외부로 보내지 않아도 되므로 규제가 엄격한 환경에서 선호됩니다.
반면 복잡한 다단계 추론이나 희귀 지식 질의에서는 대형 모델에 뒤처집니다.
따라서 작업 난이도에 따라 대형 모델과 소형 모델을 혼합해 쓰는 전략이 일반적입니다."""),
("transformer.txt", "아키텍처", """트랜스포머는 셀프 어텐션을 기반으로 한 신경망 아키텍처입니다.
2017년 인코더-디코더 구조로 제안되었으며 순환 신경망을 대체했습니다.
GPT 계열은 디코더만 사용하고 BERT 계열은 인코더만 사용합니다.
디코더 전용 구조는 이전 토큰만 참조하는 인과적 마스킹을 적용합니다.
포지셔널 인코딩으로 토큰의 순서 정보를 벡터에 주입합니다.
최근에는 회전 위치 임베딩(RoPE)이 긴 문맥 확장에 널리 쓰입니다.
잔차 연결과 층 정규화가 깊은 층을 안정적으로 학습시키는 역할을 합니다."""),
("attention.txt", "아키텍처", """셀프 어텐션은 Query, Key, Value 세 행렬의 곱으로 토큰 간 관계를 계산합니다.
Query와 Key의 내적으로 유사도를 구한 뒤 소프트맥스를 취해 가중치를 만듭니다.
멀티헤드 어텐션은 여러 부분공간에서 병렬로 어텐션을 수행해 표현력을 높입니다.
어텐션의 계산량은 시퀀스 길이의 제곱에 비례해 긴 입력에서 병목이 됩니다.
Flash Attention은 GPU 메모리 접근 패턴을 최적화해 긴 시퀀스 학습 속도를 크게 높입니다.
그룹 쿼리 어텐션(GQA)은 Key와 Value 헤드를 공유해 추론 메모리를 줄입니다.
슬라이딩 윈도우 어텐션은 참조 범위를 제한해 계산량을 선형에 가깝게 낮춥니다."""),
("finetuning.txt", "파인튜닝", """파인튜닝은 사전학습된 모델을 특정 작업에 맞게 추가 학습하는 방법입니다.
Full Fine-Tuning은 모든 파라미터를 업데이트하며 가장 높은 성능을 낼 수 있습니다.
대신 모델 크기만큼의 옵티마이저 상태가 필요해 리소스 요구가 매우 큽니다.
학습률, 배치 크기, 에폭 수 등 하이퍼파라미터 설정이 결과를 크게 좌우합니다.
데이터가 적으면 과적합이 쉽게 발생하므로 검증 손실을 함께 모니터링해야 합니다.
지시 데이터셋의 품질이 수량보다 중요하다는 것이 여러 연구의 공통된 결론입니다.
도메인 지식 주입이 목적이라면 파인튜닝보다 RAG가 더 저렴한 경우가 많습니다."""),
("peft.txt", "파인튜닝", """LoRA는 저랭크 행렬 분해로 학습 파라미터 수를 대폭 줄이는 PEFT 기법입니다.
원본 가중치는 얼려 두고 작은 어댑터 행렬 두 개만 학습해 저장 용량을 아낍니다.
QLoRA는 4bit 양자화와 LoRA를 결합해 메모리 요구를 한층 더 낮춥니다.
덕분에 RTX 4060 같은 소비자급 GPU에서도 7B 모델 학습이 가능해집니다.
랭크 r과 알파 값이 학습 용량을 결정하며 보통 8에서 64 사이를 사용합니다.
어댑터, 프리픽스 튜닝, IA3 등도 대표적인 PEFT 계열 기법입니다.
학습된 어댑터는 수십 MB에 불과해 작업별로 갈아 끼우기 쉽습니다."""),
("quantization.txt", "경량화", """양자화는 모델 가중치를 저정밀도로 변환해 메모리 사용량을 줄이는 기법입니다.
FP16에서 INT8이나 INT4로 낮추면 모델 크기가 절반 또는 4분의 1로 줄어듭니다.
GPTQ와 AWQ는 대표적인 사후 양자화 방식으로 재학습 없이 적용할 수 있습니다.
GGUF는 CPU 추론에 널리 쓰이는 파일 포맷이며 llama.cpp 계열에서 표준으로 쓰입니다.
비트를 낮출수록 메모리는 줄지만 정확도 손실이 발생할 수 있습니다.
활성값까지 양자화하면 속도 이득이 커지지만 품질 저하 위험도 함께 커집니다.
실무에서는 4bit 양자화가 품질과 비용의 균형점으로 가장 많이 선택됩니다."""),
("distillation.txt", "경량화", """지식 증류는 큰 교사 모델의 출력을 작은 학생 모델이 모방하도록 학습시키는 기법입니다.
정답 레이블 대신 교사의 확률 분포를 학습해 더 풍부한 정보를 전달받습니다.
가지치기는 중요도가 낮은 가중치를 제거해 모델 크기를 줄이는 방법입니다.
구조적 가지치기는 헤드나 층 단위로 제거해 실제 추론 속도까지 개선합니다.
증류와 양자화를 함께 적용하면 경량화 효과가 곱해집니다.
다만 압축을 과하게 하면 희귀 지식과 다단계 추론 능력이 먼저 손상됩니다.
압축 후에는 반드시 원본 모델과 같은 평가셋으로 성능 저하 폭을 측정해야 합니다."""),
("vectordb.txt", "벡터DB", """벡터 데이터베이스는 고차원 벡터를 저장하고 유사도 검색을 수행하는 시스템입니다.
전통적인 관계형 DB가 정확 일치를 찾는다면 벡터 DB는 의미가 가까운 것을 찾습니다.
ChromaDB는 오픈소스 임베디드 벡터 DB로 설치가 간단해 프로토타입에 적합합니다.
FAISS는 Meta가 만든 고속 유사도 검색 라이브러리이며 GPU 가속을 지원합니다.
Pinecone은 인프라 관리가 필요 없는 관리형 클라우드 서비스입니다.
Weaviate는 GraphQL 기반 검색 엔진으로 하이브리드 검색을 기본 제공합니다.
Milvus는 분산 아키텍처로 10억 개 이상의 벡터를 다룰 수 있습니다."""),
("ann_index.txt", "벡터DB", """근사 최근접 이웃 검색은 정확도를 조금 희생하고 속도를 얻는 검색 방식입니다.
HNSW는 계층적 그래프를 타고 이웃을 따라 내려가며 탐색하는 알고리즘입니다.
efSearch 값을 키우면 탐색 폭이 넓어져 정확도가 오르고 속도가 느려집니다.
IVF는 벡터를 클러스터로 나눈 뒤 가까운 클러스터만 탐색하는 방식입니다.
IVF에서는 nprobe 값으로 탐색할 클러스터 수를 정해 정확도를 조절합니다.
PQ는 벡터를 부분 공간으로 쪼개 압축해 메모리를 크게 절감합니다.
완전 탐색은 항상 정확하지만 데이터가 커지면 현실적으로 쓸 수 없습니다."""),
("embedding.txt", "임베딩", """임베딩은 텍스트를 고차원 벡터로 변환해 의미를 수치화하는 과정입니다.
의미가 비슷한 문장은 벡터 공간에서 가까이 위치하도록 학습됩니다.
문장 임베딩 모델은 대조학습으로 유사 쌍은 당기고 비유사 쌍은 밀어냅니다.
한국어에는 KoSimCSE나 multilingual-e5 같은 한국어 지원 모델이 적합합니다.
영어 전용 모델을 한국어에 쓰면 검색 품질이 급격히 무너집니다.
임베딩 차원이 클수록 표현력이 좋지만 저장 공간과 검색 비용이 함께 늘어납니다.
질의용과 문서용 프리픽스를 구분해 넣어야 하는 모델도 있으니 문서를 확인해야 합니다."""),
("similarity.txt", "임베딩", """코사인 유사도는 두 벡터의 각도로 유사도를 측정하며 1에 가까울수록 유사합니다.
내적은 벡터의 크기까지 반영하므로 길이가 다른 벡터에서는 결과가 달라집니다.
유클리드 거리는 값이 작을수록 유사하다는 점에서 방향이 반대입니다.
벡터를 정규화하면 코사인과 내적이 같아지고 L2 거리도 같은 순위를 냅니다.
서로 다른 척도로 만든 점수를 그대로 비교하면 잘못된 결론에 이릅니다.
검색기를 비교할 때는 반드시 같은 거리 척도 위에 올려놓아야 합니다.
BM25 점수와 코사인 점수는 스케일이 달라 단순 합산이 불가능합니다."""),
("prompt_eng.txt", "프롬프트", """프롬프트 엔지니어링은 원하는 출력을 얻기 위한 입력 설계 기술입니다.
같은 모델이라도 입력을 어떻게 쓰느냐에 따라 출력 품질이 크게 달라집니다.
제로샷은 예시 없이 지시만 주고 퓨샷은 몇 개의 입출력 예시를 함께 제공합니다.
예시는 형식을 알려주는 역할이 크므로 원하는 출력 형태를 그대로 보여주는 것이 좋습니다.
시스템 프롬프트로 역할, 제약조건, 응답 형식을 명시적으로 정의합니다.
부정 지시보다 긍정 지시가 더 잘 지켜지는 경향이 있습니다.
출력 형식을 JSON 등으로 고정하면 후처리가 안정적이 됩니다."""),
("cot.txt", "프롬프트", """Chain-of-Thought는 모델이 단계별 추론 과정을 서술하도록 유도하는 기법입니다.
정답만 바로 내놓게 하는 대신 중간 과정을 쓰게 하면 정답률이 오릅니다.
복잡한 수리 문제와 논리 문제에서 특히 효과가 큽니다.
차근차근 생각해보자는 짧은 문구만 넣어도 효과가 나타나는 경우가 있습니다.
Self-Consistency는 여러 추론 경로를 뽑아 다수결로 최종 답을 정합니다.
Tree-of-Thought는 여러 갈래를 탐색하며 유망한 경로를 확장합니다.
추론 과정이 길어지면 토큰 비용과 지연이 함께 늘어난다는 단점이 있습니다."""),
("rl_methods.txt", "강화학습", """강화학습은 에이전트가 환경과 상호작용하며 보상을 최대화하도록 학습하는 방법입니다.
정책은 상태에서 행동을 고르는 규칙이며 학습의 대상이 됩니다.
PPO는 정책이 한 번에 크게 변하지 않도록 제한해 안정적인 업데이트를 보장합니다.
클리핑을 통해 이전 정책과의 비율을 제한하는 것이 PPO의 핵심 아이디어입니다.
GRPO는 DeepSeek이 제안한 효율적인 정책 최적화 방법입니다.
GRPO는 별도의 가치 함수 없이 그룹 내 상대 비교로 이점을 추정해 메모리를 절약합니다.
언어 모델 학습에서는 생성된 응답 전체를 하나의 행동으로 보는 경우가 많습니다."""),
("alignment.txt", "강화학습", """정렬은 모델의 출력을 사람이 바라는 방향으로 맞추는 작업입니다.
RLHF는 인간의 선호도를 보상 모델로 학습한 뒤 강화학습으로 정책에 반영합니다.
먼저 사람이 두 응답 중 나은 것을 고르게 해 선호 데이터를 모읍니다.
그 데이터로 보상 모델을 학습하고 PPO로 정책을 업데이트하는 3단계 구조입니다.
DPO는 별도의 보상 모델 없이 선호 쌍으로 정책을 직접 최적화합니다.
구현이 단순하고 학습이 안정적이어서 최근 널리 쓰입니다.
이런 정렬 기법은 모델의 안전성과 유용성을 높이고 유해한 출력을 줄이는 데 사용됩니다."""),
("rag_basic.txt", "RAG", """RAG는 외부 지식을 검색해 LLM 답변에 근거를 제공하는 기술입니다.
모델의 파라미터에 지식을 넣는 대신 필요할 때 찾아 쓰는 방식입니다.
문서 로딩, 청킹, 임베딩, 검색, 생성의 다섯 단계로 구성됩니다.
파인튜닝 없이 최신 정보를 반영할 수 있다는 것이 가장 큰 장점입니다.
근거 문서를 함께 제시할 수 있어 답변의 출처를 추적할 수 있습니다.
지식이 바뀌면 문서만 교체하면 되므로 유지보수 비용이 낮습니다.
반면 검색이 실패하면 아무리 좋은 LLM도 옳은 답을 낼 수 없습니다."""),
("chunking.txt", "RAG", """청킹은 긴 문서를 검색 단위로 쪼개는 과정이며 RAG 품질의 출발점입니다.
청크가 너무 크면 무관한 내용이 섞여 벡터가 흐려집니다.
너무 작으면 문맥이 끊겨 그 자체로는 의미를 알 수 없는 조각이 됩니다.
오버랩을 두면 경계에서 잘린 문맥을 어느 정도 보완할 수 있습니다.
문단이나 제목 같은 문서 구조를 경계로 삼으면 품질이 좋아집니다.
표와 코드 블록은 중간에서 자르면 의미가 파괴되므로 따로 처리해야 합니다.
적정 크기는 도메인마다 다르므로 실제 질의로 측정해 정하는 것이 맞습니다."""),
("advanced_rag.txt", "RAG", """Advanced RAG는 기본 검색의 약점을 보완하는 기법들의 묶음입니다.
HyDE는 질문으로 가상 답변 문서를 생성한 뒤 그 문서로 검색하는 기법입니다.
질문과 문서의 형태 차이를 줄여 문서 대 문서 비교로 바꾸는 것이 핵심입니다.
리랭킹은 1차 검색 결과를 교차 인코더로 다시 채점해 순서를 바로잡습니다.
교차 인코더는 질문과 문서를 함께 입력해 관련성을 직접 예측합니다.
Parent Document Retriever는 작은 청크로 찾고 큰 부모 청크를 반환합니다.
쿼리 재작성과 컨텍스트 압축도 자주 함께 사용되는 기법입니다."""),
("hybrid_search.txt", "RAG", """하이브리드 검색은 BM25 같은 키워드 검색과 벡터 검색을 결합하는 방식입니다.
BM25는 단어 빈도와 문서 길이를 반영해 점수를 매기는 고전적 검색 알고리즘입니다.
고유명사나 약어, 코드처럼 정확한 토큰 일치가 중요한 질의에서 특히 강합니다.
반대로 표현이 다르면 못 찾는다는 약점이 있어 벡터 검색으로 보완합니다.
RRF는 두 순위 목록을 상호 순위 역수로 융합하는 방법입니다.
점수 스케일이 다른 검색기를 순위만으로 합칠 수 있다는 것이 장점입니다.
가중치는 도메인과 질의 유형에 따라 실측으로 정해야 합니다."""),
("rag_eval.txt", "평가", """RAG 평가는 검색 품질과 생성 품질을 나누어 측정해야 합니다.
검색 품질은 정답 문서를 얼마나 잘 올렸는지를 Hit@k와 MRR로 잽니다.
RAGAS는 RAG 파이프라인을 지표화해 자동 평가하는 프레임워크입니다.
Faithfulness는 답변이 근거 문서에 충실한지를 봅니다.
Answer Relevancy는 답변이 질문에 실제로 대응하는지를 봅니다.
Context Precision과 Context Recall은 검색된 컨텍스트의 질을 따로 측정합니다.
평가 지표를 검색기의 목적함수와 같게 만들면 베이스라인이 항상 이기게 됩니다."""),
("hallucination.txt", "평가", """환각은 모델이 근거 없는 내용을 사실처럼 생성하는 현상입니다.
그럴듯한 문체로 서술되기 때문에 사람이 눈으로 걸러내기 어렵습니다.
검색된 근거를 제시하도록 프롬프트를 설계하면 환각을 줄일 수 있습니다.
근거에 없으면 모른다고 답하게 하는 것이 실무의 기본 전략입니다.
답변의 각 문장을 근거 문서와 대조해 검증하는 후처리도 사용됩니다.
검색이 실패했을 때 억지로 답하지 않게 하는 임계값 설정도 효과적입니다.
환각률은 도메인 전문가가 만든 평가셋으로 주기적으로 측정해야 합니다."""),
("agent.txt", "에이전트", """LLM 에이전트는 도구를 호출하며 여러 단계로 문제를 해결하는 시스템입니다.
한 번의 생성으로 끝내지 않고 관찰과 행동을 반복하는 것이 특징입니다.
ReAct는 추론과 행동을 번갈아 수행하는 대표적인 에이전트 패턴입니다.
계획 수립과 실행을 분리하는 Plan-and-Execute 패턴도 널리 쓰입니다.
LangGraph는 상태 기반 그래프로 에이전트의 흐름을 명시적으로 제어합니다.
반복 횟수 제한과 실패 처리를 두지 않으면 무한 루프에 빠질 수 있습니다.
에이전트는 강력하지만 지연과 비용이 커서 단순 작업에는 과합니다."""),
("tool_use.txt", "에이전트", """함수 호출은 모델이 정해진 스키마에 맞춰 도구 인자를 생성하는 기능입니다.
모델은 도구를 직접 실행하지 않고 어떤 도구를 어떤 인자로 부를지만 출력합니다.
실행은 애플리케이션이 담당하고 그 결과를 다시 모델에 넣어 최종 답변을 만듭니다.
도구 설명과 파라미터 설명을 명확히 쓸수록 호출 정확도가 올라갑니다.
MCP는 모델과 외부 도구를 연결하는 표준 프로토콜입니다.
표준을 쓰면 도구를 여러 애플리케이션에서 재사용할 수 있습니다.
도구 실행에는 권한 검증과 실패 처리를 반드시 함께 설계해야 합니다."""),
]


documents = [
    Document(page_content=content, metadata={"source": src, "topic": topic})
    for src, topic, content in RAW_DOCS
]

# --- 정답표: 질문 → "이 문서에 답이 있다"를 사람이 직접 적어둔 것 ---
# 정답을 리스트로 둔 이유: 답이 여러 문서에 걸칠 수 있어서
BENCHMARK = [
    ("소형 언어 모델(sLLM)이 뭐야?",                    ["slm.txt"]),
    ("GPU 메모리가 적어도 큰 모델을 학습할 수 있는 방법은?", ["peft.txt"]),
    ("LLM을 더 안전하게 만드는 학습 방법은?",              ["alignment.txt"]),
    ("벡터 검색에 사용할 수 있는 오픈소스 도구들은?",        ["vectordb.txt"]),
    ("AI 모델의 출력 품질을 높이는 입력 기법은?",           ["prompt_eng.txt", "cot.txt"]),
    ("GRPO가 뭐야?",                                 ["rl_methods.txt"]),
    ("RTX 4060으로도 학습이 되나?",                     ["peft.txt"]),
    ("GraphQL을 쓰는 검색 엔진은?",                     ["vectordb.txt"]),
    ("단계별로 생각하게 시키면 정답률이 오르나?",            ["cot.txt"]),
    ("사람의 선호를 반영해서 모델을 고치는 기법은?",          ["alignment.txt"]),
    ("모델이 없는 사실을 지어내는 걸 뭐라고 하나?",           ["hallucination.txt"]),
    ("문서를 얼마나 잘게 쪼개야 하나?",                    ["chunking.txt"]),
    ("키워드 검색과 벡터 검색을 같이 쓰는 방법은?",           ["hybrid_search.txt"]),
    ("긴 문장을 빠르게 학습시키는 어텐션 최적화는?",          ["attention.txt"]),
    ("CPU에서 모델을 돌릴 때 쓰는 파일 포맷은?",            ["quantization.txt"]),
    ("검색 결과 순서를 바로잡는 기법은?",                  ["advanced_rag.txt"]),
    ("RAG 파이프라인을 수치로 평가하는 도구는?",            ["rag_eval.txt"]),
    ("모델이 도구를 직접 호출하게 하려면?",                 ["tool_use.txt"]),
    ("근사 최근접 이웃에서 정확도를 조절하는 값은?",          ["ann_index.txt"]),
    ("한국어 문장 임베딩에 쓸 만한 모델은?",                ["embedding.txt"]),
    # --- 유형당 10개를 맞추기 위한 추가 질문 ---
    ("RoPE가 뭐야?",                                ["transformer.txt"]),
    ("GQA는 왜 쓰나?",                              ["attention.txt"]),
    ("GPTQ와 AWQ는 무슨 방식이야?",                    ["quantization.txt"]),
    ("ReAct 패턴이 뭐야?",                           ["agent.txt"]),
    ("MCP는 무엇을 위한 규약이야?",                      ["tool_use.txt"]),
    ("큰 모델의 출력을 흉내 내게 해서 작은 모델을 만드는 방법은?",   ["distillation.txt"]),
    ("두 문장이 얼마나 비슷한지 숫자로 재려면?",              ["similarity.txt"]),
    ("지식을 모델에 넣지 않고 필요할 때 찾아 쓰게 하려면?",       ["rag_basic.txt"]),
    ("답을 여러 갈래로 뽑아 다수결로 정하는 방법은?",           ["cot.txt"]),
    ("파인튜닝할 때 무엇을 설정해야 하나?",                  ["finetuning.txt"]),
]

test_questions = [q for q, _ in BENCHMARK]
gold_sources = {q: set(g) for q, g in BENCHMARK}

# --- 청킹 ---
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
splits = text_splitter.split_documents(documents)

print("📄 시험지 준비 완료")
print(f"  문서 {len(documents)}개를 {len(splits)}개 조각으로 나눔")
print(f"  질문 {len(test_questions)}개, 각 질문마다 정답 문서를 표시해 둠")
print(f"  3개를 뽑는 것은 전체 조각의 {3 / len(splits):.0%} — 적당히 어려운 난이도")

In [ ]:
# 📏 채점 도구 — 사람이 만든 정답표와 대조한다 (유사도 점수는 쓰지 않는다)
import time
import numpy as np


def evaluate_retriever(retriever, name, questions=None, k=3, verbose=True):
    """검색기를 정답표로 채점한다.

    재는 것은 세 가지뿐이다.
      Hit@1  : 1등으로 가져온 문서가 정답인 질문의 비율
      Hit@3  : 정답이 3개 안에 든 질문의 비율
      검색 시간: 질문 1개를 처리하는 데 걸린 평균 시간(ms)
    """
    questions = questions or test_questions
    rows, hits1, hits3, times = [], [], [], []

    for q in questions:
        start = time.time()
        docs = retriever.invoke(q)
        elapsed = time.time() - start

        srcs = [d.metadata.get("source", "?") for d in docs[:k]]
        gold = gold_sources[q]
        hit1 = bool(srcs) and srcs[0] in gold          # 1등이 정답인가
        hit3 = any(s in gold for s in srcs)            # 3개 안에 정답이 있는가

        hits1.append(hit1); hits3.append(hit3); times.append(elapsed)
        rows.append({"question": q, "sources": srcs, "gold": gold,
                     "hit1": hit1, "hit3": hit3, "elapsed": elapsed})

    result = {
        "name": name,
        "hit1": float(np.mean(hits1)),
        "hit3": float(np.mean(hits3)),
        "time": float(np.mean(times)) * 1000,          # ms
        "rows": rows,
    }

    if verbose:
        print(f"\n📊 [{name}]")
        print(f"  Hit@1 {result['hit1']:.0%}  |  Hit@3 {result['hit3']:.0%}  |  "
              f"검색 시간 {result['time']:.0f}ms")
        misses = [r for r in rows if not r["hit1"]]
        if misses:
            print(f"  ❌ 1등을 놓친 질문 {len(misses)}개 (앞 3개만 표시):")
            for r in misses[:3]:
                print(f"     · {r['question']}")
                print(f"       정답={sorted(r['gold'])} / 검색={r['sources']}")
    return result


print("📏 채점 도구 준비 완료")
print("  evaluate_retriever(검색기, 이름) → Hit@1 / Hit@3 / 검색 시간")
print()
print("  Hit@1   : 1등으로 가져온 문서가 정답인 질문의 비율")
print("  Hit@3   : 정답이 3개 안에 든 질문의 비율")
print("  검색 시간: 질문 1개당 평균 소요 시간")
print()
print("  채점에 유사도 점수는 전혀 쓰지 않습니다.")
print("  사람이 적어둔 정답 문서와 파일 이름을 맞춰볼 뿐입니다.")

---

## 3️⃣ 기본 RAG 점수 재기 (기준선)

여기서 나온 점수가 앞으로 모든 기법의 비교 대상입니다.

### 그 전에 — 임베딩 모델부터 확인하세요

한국어 문서에 **영어 전용 모델**을 쓰면 검색이 거의 찍기 수준이 됩니다.
같은 시험지에서 모델만 바꿔 재봤습니다.

| 임베딩 모델 | | Hit@1 | Hit@3 |
|-------------|---|-------|-------|
| `all-MiniLM-L6-v2` | 영어 전용 | **17%** | 27% |
| `BM-K/KoSimCSE-roberta-multitask` | 한국어 됨 | **60%** | 83% |

문서가 24개니까 **아무거나 찍어도 4%** 는 나옵니다. 영어 모델의 17%는 거기서 조금 나은 정도,
즉 **검색이 사실상 작동하지 않는 상태**입니다.

> 🔑 기법 하나 추가해서 얻는 이득보다, **임베딩 모델을 바꿔서 얻는 이득이 훨씬 큽니다.**
> Advanced RAG를 고민하기 전에 여기부터 보세요.

Session 03에서 쓴 것과 같은 KoSimCSE 모델을 씁니다.


In [ ]:
# 🔧 임베딩 · 벡터스토어 · LLM 설정
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ⚠️ 한국어 코퍼스에는 반드시 한국어를 아는 임베딩 모델을 쓴다.
#    all-MiniLM-L6-v2 같은 영어 전용 모델을 쓰면 검색이 거의 무작위가 되고,
#    그 위에 어떤 Advanced 기법을 얹어도 효과가 나타나지 않는다.
EMBEDDING_MODEL = "BM-K/KoSimCSE-roberta-multitask"   # Session 03과 동일

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},   # 정규화 → 코사인=내적
)
print(f"✅ 임베딩 모델: {EMBEDDING_MODEL}")

# 벡터스토어 (코사인 거리)
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="advanced_rag_demo",
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"✅ 벡터스토어 구축 완료 ({vectorstore._collection.count()}개 청크)")

# 기본 Retriever
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# LLM (HyDE 섹션에서만 사용)
USE_OLLAMA = False
llm = None
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

if USE_OLLAMA:
    from langchain_community.llms import Ollama
    llm = Ollama(model="qwen2.5:1.5b", temperature=0, num_ctx=2048)
    print("✅ Ollama LLM 설정 완료")
elif os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    print("✅ OpenAI LLM 설정 완료 (gpt-4o-mini)")
else:
    print("⚠️ OPENAI_API_KEY 없음 — HyDE 섹션은 건너뜁니다 (나머지는 정상 동작)")

In [ ]:
# 📊 기준 점수 재기 — 앞으로 모든 기법을 이 값과 비교한다
base_result = evaluate_retriever(base_retriever, "기본 RAG")

# 이 점수가 기준선이다
print("\n" + "=" * 70)
print("이 숫자가 기준선입니다. 이제 기법을 하나씩 얹으며 이 값을 넘는지 봅니다.")
print("=" * 70)

---

## 4️⃣ HyDE — 가짜 답변을 만들어서 검색하기

```
기본 RAG : 질문  ─────────────────────→ 문서 찾기
HyDE    : 질문 → LLM이 가짜 답변 작성 → 그 답변으로 문서 찾기
```

질문은 짧고 의문형인데 문서는 길고 서술형이라 모양이 안 맞습니다.
가짜라도 **답변 모양**으로 바꿔서 찾으면 더 잘 맞을 것이다 — 이게 HyDE의 발상입니다.

### 잘 안 되는 경우

LLM이 **모르는 용어면 엉뚱한 분야 얘기를 지어냅니다.** 그 가짜 문서로 찾으니 결과도 같이 틀립니다.
아래 셀에서 직접 보세요 — "GRPO"를 물으면 gpt-4o-mini가 이걸 **공급망 관리 서류**로 착각합니다.

그리고 **질문마다 LLM을 한 번씩 부르므로 아주 느립니다.** 기본 RAG가 15ms인데 HyDE는 약 1,400ms입니다.

> ⚠️ HyDE는 돌릴 때마다 점수가 달라집니다. `temperature=0`이어도 생성 결과가 완전히 같지는 않아서,
> 같은 코드를 두 번 돌리면 Hit@1이 60~67% 사이를 오갑니다.
> **좋아진 폭이 이 흔들림보다 작으면, 좋아진 게 아닙니다.**
> 여러분 화면의 숫자도 아래 해설과 다를 수 있는데, 그 사실 자체가 이 기법에 대한 결론입니다.

> 💡 이 섹션만 OpenAI API 키가 필요합니다. 없으면 셀이 알아서 건너뜁니다.


In [ ]:
# 🔮 HyDE Retriever 구현
from langchain.prompts import PromptTemplate

hyde_prompt = PromptTemplate(
    template="""다음 질문에 대한 답변을 기술 문서의 한 문단처럼 3문장 이내로 작성하세요.
사실 여부는 중요하지 않습니다. 질문에 등장하지 않은 관련 전문 용어를 포함하세요.

질문: {question}

답변 문단:""",
    input_variables=["question"],
)


class HyDERetriever:
    """HyDE: 질문 → LLM이 가상 답변 생성 → 그 답변으로 벡터 검색"""

    def __init__(self, llm, vectorstore, k=3):
        self.llm = llm
        self.vectorstore = vectorstore
        self.k = k
        self.last_hypothetical = None      # 관찰용: 마지막으로 생성한 가상 문서

    def invoke(self, query):
        # 1. LLM으로 가상 답변 문서 생성
        out = self.llm.invoke(hyde_prompt.format(question=query))
        hypothetical = out if isinstance(out, str) else out.content
        self.last_hypothetical = hypothetical.strip()

        # 2. 질문이 아니라 '가상 문서'로 검색한다 — 이것이 HyDE의 전부
        return self.vectorstore.similarity_search(self.last_hypothetical, k=self.k)

    def get_relevant_documents(self, query):
        return self.invoke(query)


if llm is None:
    hyde_retriever = None
    print("⚠️ LLM이 없어 HyDE를 건너뜁니다.")
else:
    hyde_retriever = HyDERetriever(llm, vectorstore, k=3)
    print("✅ HyDE Retriever 생성 완료")
    print("  📊 동작: 질문 → LLM(가상 문서 생성) → 벡터 검색")

In [ ]:
# 🔍 HyDE가 만드는 가상 문서를 직접 확인 — 이 기법의 성패가 여기서 갈린다
if hyde_retriever is None:
    print("⚠️ LLM 없음 — 건너뜁니다.")
else:
    for q in ["GRPO가 뭐야?", "CPU에서 모델을 돌릴 때 쓰는 파일 포맷은?"]:
        docs = hyde_retriever.invoke(q)
        print(f"❓ 질문: {q}")
        print(f"🔮 LLM이 지어낸 가상 문서:")
        print(f"   {hyde_retriever.last_hypothetical[:220]}...")
        print(f"📄 그 문서로 검색한 결과: {[d.metadata['source'] for d in docs]}")
        print(f"✅ 정답: {sorted(gold_sources[q])}")
        print("-" * 70)

    print("\n💡 LLM이 도메인을 모르면 엉뚱한 분야의 문서를 지어냅니다.")
    print("   그 오염된 벡터로 검색하므로 결과가 함께 오염됩니다.")

In [ ]:
# 📊 HyDE 평가
if hyde_retriever is None:
    hyde_result = None
    print("⚠️ LLM 없음 — HyDE 평가를 건너뜁니다.")
else:
    hyde_result = evaluate_retriever(hyde_retriever, "HyDE")
    print(f"\n  기본 RAG 대비 Hit@1: {base_result['hit1']:.0%} → {hyde_result['hit1']:.0%}")

---

## 5️⃣ Reranking — 순서를 다시 매기기

```
1단계: 빠른 검색으로 후보 10개 (넓게)
2단계: 똑똑한 모델로 다시 채점 → 3개 (좁게)
```

- **1단계(Bi-encoder)**: 질문과 문서를 **따로** 벡터로 만들어 비교. 미리 계산해둘 수 있어 빠르지만 덜 정확
- **2단계(Cross-encoder)**: 질문과 문서를 **붙여서** 모델에 넣고 점수를 직접 뽑음. 느리지만 정확

2단계 모델은 문서마다 한 번씩 돌려야 해서 전체 문서에는 못 씁니다.
그래서 1단계로 후보를 줄인 뒤 2단계로 정밀 채점합니다.

### 여기도 모델이 전부입니다

리랭킹 모델만 바꿔서 재봤습니다.

| Cross-encoder 모델 | | Hit@1 | Hit@3 |
|--------------------|---|-------|-------|
| 안 씀 (기본 RAG) | | 60% | 83% |
| `cross-encoder/ms-marco-MiniLM-L-6-v2` | 영어 전용 | **53%** | 73% |
| `BAAI/bge-reranker-base` | 다국어 (작음) | 57% | 70% |
| `BAAI/bge-reranker-v2-m3` | 다국어 | **93%** | **97%** |

**영어 모델을 쓰면 안 쓰느니만 못합니다.** 순서를 바로잡는 게 아니라 마구 섞는 셈이거든요.
같은 "리랭킹"인데 모델 하나로 53%와 93%가 갈립니다.
**기법이 효과 없어 보이면, 기법 말고 모델을 먼저 의심하세요.**


In [ ]:
# 🎯 Reranking Retriever 구현
from sentence_transformers import CrossEncoder

# ⚠️ 리랭커도 한국어를 알아야 한다.
#    cross-encoder/ms-marco-MiniLM-L-6-v2 는 영어 전용이라 한국어에서는
#    기본 RAG보다 오히려 나쁘다 (과제 2번에서 직접 확인).
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"   # 다국어 cross-encoder (약 2.2GB)

print(f"🔧 Cross-encoder 로딩 중: {RERANKER_MODEL}")
print("   (최초 실행 시 다운로드에 시간이 걸립니다)")
reranker_model = CrossEncoder(RERANKER_MODEL, max_length=512, device=DEVICE)
print("✅ Reranker 로딩 완료")


class RerankingRetriever:
    """2단계 검색: 넓게 뽑고(bi-encoder) → 정밀하게 재채점(cross-encoder)"""

    def __init__(self, vectorstore, reranker, top_k=3, fetch_k=10):
        self.vectorstore = vectorstore
        self.reranker = reranker
        self.top_k = top_k
        self.fetch_k = fetch_k          # 1단계에서 뽑을 후보 수

    def invoke(self, query):
        # 1단계: 빠른 벡터 검색으로 후보를 넓게 확보
        candidates = self.vectorstore.similarity_search(query, k=self.fetch_k)
        if not candidates:
            return []

        # 2단계: (질문, 문서) 쌍마다 cross-encoder로 관련성 점수 산출
        pairs = [[query, doc.page_content] for doc in candidates]
        scores = self.reranker.predict(pairs)

        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in ranked[: self.top_k]]

    def get_relevant_documents(self, query):
        return self.invoke(query)


reranking_retriever = RerankingRetriever(vectorstore, reranker_model, top_k=3, fetch_k=10)
print("✅ Reranking Retriever 생성 완료")
print("  📊 동작: 벡터 검색 10개 → cross-encoder 재채점 → 상위 3개")

# 리랭킹이 의미가 있으려면 1단계 후보 안에 정답이 들어 있어야 한다. 먼저 그것부터 확인한다.
pool_hits = 0
for q in test_questions:
    pool = vectorstore.similarity_search(q, k=10)
    if any(d.metadata["source"] in gold_sources[q] for d in pool):
        pool_hits += 1
print(f"\n🔍 1단계 후보(top-10)에 정답이 포함된 비율: {pool_hits}/{len(test_questions)} "
      f"= {pool_hits/len(test_questions):.0%}")
print("   → 이 값이 리랭킹으로 도달 가능한 Hit@1의 상한이다.")
print("     정답은 이미 후보 안에 있다. 문제는 '순서'였다.")

In [ ]:
# 📊 Reranking 평가
reranking_result = evaluate_retriever(reranking_retriever, "Reranking")
print(f"\n  기본 RAG 대비  Hit@1 {base_result['hit1']:.0%} → {reranking_result['hit1']:.0%}"
      f"  |  Hit@3 {base_result['hit3']:.0%} → {reranking_result['hit3']:.0%}")

---

## 6️⃣ Ensemble — 키워드 검색과 의미 검색 합치기

| | BM25 (키워드) | 의미 검색 |
|---|---|---|
| 찾는 방식 | 단어가 글자 그대로 겹치는지 | 뜻이 비슷한지 |
| 강한 곳 | 약어, 제품명, 코드 | 표현이 달라도 찾음 |
| 약한 곳 | 단어가 다르면 못 찾음 | 희귀한 약어를 못 알아봄 |

**약점이 서로 반대**라서 합치면 서로 메워줄 거라는 게 앙상블의 논리입니다.

### 어떻게 합치나 — RRF

두 검색기의 **점수**는 단위가 달라서 그냥 못 더합니다.

```
의미 검색 : 0.71   (0~1)
BM25     : 8.34   (상한 없음)
```

그래서 점수를 버리고 **등수만** 씁니다.

```
RRF 점수 = Σ  가중치 / (60 + 등수)
```

1등이면 `가중치/61`, 2등이면 `가중치/62` … 이렇게 바꿔서 더합니다.
**두 검색기 모두에서 나온 문서는 두 번 더해지므로** 자연스럽게 위로 올라갑니다.

### ⚠️ 가중치를 0.5 / 0.5로 두지 마세요

가중치가 같으면 양쪽 1등의 점수가 **소수점까지 똑같아집니다.**

```
BM25  1등 : 0.5 / 61 = 0.008197
의미  1등 : 0.5 / 61 = 0.008197    ← 완전히 같음
```

동점이면 파이썬은 **먼저 들어온 쪽**을 앞에 둡니다. LangChain은 `retrievers=[...]`에 적은 순서대로
넣으므로, **먼저 적은 검색기가 동점을 전부 가져갑니다.**

실제로 `weights=[0.5, 0.5]`로 두면 Ensemble의 1등이 BM25의 1등과 **30개 중 27개에서 같았습니다.**
합쳐진 게 아니라 BM25가 통째로 채택된 겁니다. (가중치를 다르게 주면 16개로 떨어집니다.)

같은 질문 30개를 다섯 가지 설정으로 돌려봤습니다.
숫자는 **Hit@1** — 1등으로 가져온 문서가 정답인 질문의 비율입니다.

| 어떤 설정인가 | 1등이 정답인 비율 |
|---------------|------------------|
| ① 의미 검색 **하나만** 씀 | 60% |
| ② BM25 **하나만** 씀 | 57% |
| ③ 둘을 **반반(50:50)** 으로 합침 — 코드에 **BM25를 먼저** 적음 | 60% |
| ④ 둘을 **반반(50:50)** 으로 합침 — 코드에 **의미 검색을 먼저** 적음 | **70%** |
| ⑤ 의미 검색에 **더 무게(40:60)** — 이 노트북의 설정 | **70%** |

```python
# ③ 과 ④ 의 차이는 이 한 줄이 전부입니다
EnsembleRetriever(retrievers=[bm25_retriever, base_retriever], weights=[0.5, 0.5])  # ③ 60%
EnsembleRetriever(retrievers=[base_retriever, bm25_retriever], weights=[0.5, 0.5])  # ④ 70%
```

**③과 ④는 가중치도 같고 검색기도 같습니다. 리스트에 적은 순서만 다른데 10%p 차이**가 납니다.

즉 **③을 보고 "앙상블은 효과 없네"라고 결론 내리면 틀린 것**입니다.
가중치만 다르게 주면(⑤) 제대로 동작합니다.
해결은 간단합니다 — **가중치를 다르게 줘서 동점을 없애면** 됩니다.

아래에서 RRF를 직접 계산해 라이브러리와 같은 답이 나오는지 확인하고, 이 함정도 재현해봅니다.


In [ ]:
# 🔗 Ensemble Retriever (BM25 + 시맨틱)
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# BM25 — 키워드(토큰) 일치 기반
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 3
print("✅ BM25 Retriever 생성 완료")

# ⚠️ 가중치를 0.5 / 0.5 로 두면 안 된다. 아래 셀에서 이유를 직접 확인한다.
#    RRF 점수가 정확히 같아지는 '동점'이 대량 발생하고, 동점일 때는
#    retrievers 리스트에 먼저 적은 검색기가 무조건 이긴다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, base_retriever],
    weights=[0.4, 0.6],          # BM25 40% + 시맨틱 60% — 동점을 없앤다
)
print("✅ Ensemble Retriever 생성 완료 (BM25 40% + 시맨틱 60%, RRF 융합)")

In [ ]:
# 🔬 RRF를 직접 구현해 본다 — EnsembleRetriever가 내부에서 하는 일
# LangChain은 weighted_reciprocal_rank() 안에서 아래와 똑같은 계산을 한다.
#   langchain/retrievers/ensemble.py :
#       rrf_score[doc.page_content] += weight / (rank + self.c)

RRF_C = 60      # LangChain EnsembleRetriever의 기본값 (c 파라미터)


def manual_rrf(query, retrievers, weights, c=RRF_C, top_k=3, explain=False):
    """가중 RRF 융합을 직접 계산한다.

    점수를 쓰지 않고 '순위'만 쓴다는 것이 핵심이다.
    BM25 점수(상한 없음)와 코사인 유사도(0~1)는 스케일이 달라 더할 수 없기 때문이다.
    """
    scores, seen, trace = {}, {}, {}
    for retriever, w, label in zip(retrievers, weights, ["BM25", "dense"]):
        for rank, doc in enumerate(retriever.invoke(query), start=1):
            key = doc.page_content                  # LangChain과 동일한 기준으로 문서 식별
            contrib = w / (c + rank)
            scores[key] = scores.get(key, 0.0) + contrib     # ← 두 목록에 다 나오면 '누적'
            seen[key] = doc
            trace.setdefault(key, []).append(f"{label}{rank}위 {w}/({c}+{rank})={contrib:.6f}")

    ranked = sorted(scores.items(), key=lambda x: -x[1])
    if explain:
        for key, sc in ranked:
            print(f"  {seen[key].metadata['source']:<20}{sc:.6f}   {'  +  '.join(trace[key])}")
    return [seen[k] for k, _ in ranked[:top_k]]


# --- 계산 과정 출력 ---
demo_q = "근사 최근접 이웃에서 정확도를 조절하는 값은?"
print(f"질문: {demo_q}")
print(f"정답 출처: {sorted(gold_sources[demo_q])}\n")

print("[1단계] 각 검색기의 순위 목록")
b_list = [d.metadata["source"] for d in bm25_retriever.invoke(demo_q)]
d_list = [d.metadata["source"] for d in base_retriever.invoke(demo_q)]
print(f"  {'순위':<6}{'BM25':<22}{'시맨틱(dense)':<22}")
for i in range(max(len(b_list), len(d_list))):
    print(f"  {i+1:<6}{b_list[i] if i < len(b_list) else '':<22}"
          f"{d_list[i] if i < len(d_list) else '':<22}")

print(f"\n[2단계] RRF 융합 (c={RRF_C}, 가중치 BM25 0.4 / dense 0.6)")
fused = manual_rrf(demo_q, [bm25_retriever, base_retriever], [0.4, 0.6], explain=True)

print(f"\n[3단계] 최종 top-3: {[d.metadata['source'] for d in fused]}")
lib = [d.metadata["source"] for d in ensemble_retriever.invoke(demo_q)][:3]
print(f"        EnsembleRetriever 결과: {lib}")
print(f"        일치 여부: {'✅ 같다' if [d.metadata['source'] for d in fused] == lib else '❌ 다르다'}")

In [ ]:
# ⚠️ 동점의 함정 — 가중치가 같으면 '리스트 순서'가 결과를 정한다
# 두 검색기의 가중치가 같으면 1위끼리 RRF 점수가 정확히 같아진다.
#     BM25  1위 : 0.5 / (60+1) = 0.008197
#     dense 1위 : 0.5 / (60+1) = 0.008197   ← 완전히 동일
# 파이썬 sorted()는 안정 정렬이라 동점이면 먼저 들어온 쪽이 앞선다.
# LangChain은 retrievers 리스트 순서대로 문서를 이어붙이므로,
# '먼저 적은 검색기'가 동점 승부를 전부 가져간다.

import numpy as np


def quick_hit1(retriever):
    return np.mean([retriever.invoke(q)[0].metadata["source"] in gold_sources[q]
                    for q in test_questions])




configs = [
    ("① 의미 검색 하나만",                      base_retriever),
    ("② BM25 하나만",                         bm25_retriever),
    ("③ 반반(50:50) — BM25를 먼저 적음",        EnsembleRetriever(
        retrievers=[bm25_retriever, base_retriever], weights=[0.5, 0.5])),
    ("④ 반반(50:50) — 의미 검색을 먼저 적음",     EnsembleRetriever(
        retrievers=[base_retriever, bm25_retriever], weights=[0.5, 0.5])),
    ("⑤ 의미 검색에 더 무게(40:60) ← 이 노트북",  ensemble_retriever),
]

print("같은 질문 30개를 다섯 가지 설정으로 돌린 결과")
print(f"\n{'어떤 설정인가':<44}{'1등이 정답인 비율':>16}")
print("-" * 62)
for name, ret in configs:
    print(f"{name:<42}{quick_hit1(ret):>14.0%}")

print("""
💡 ③ 과 ④ 를 비교하세요. 두 설정의 차이는 이 한 줄뿐입니다.

     ③  retrievers=[bm25_retriever, base_retriever]
     ④  retrievers=[base_retriever, bm25_retriever]

   가중치도 같고 검색기도 같은데 점수가 달라집니다.
   가중치가 0.5로 같으면 양쪽 1등의 RRF 점수가 소수점까지 똑같아지고,
   동점일 때는 먼저 적은 검색기가 전부 이기기 때문입니다.

   그래서 ③ 만 보고 '앙상블은 효과 없다'고 결론 내리면 틀립니다.
   해결은 간단합니다 — 가중치를 다르게 줘서 동점을 없애면 됩니다 (⑤).""")

In [ ]:
# 📊 BM25 단독 / Ensemble 채점 — 섞을 값어치가 있는지부터 본다
bm25_result = evaluate_retriever(bm25_retriever, "BM25만", verbose=False)
print(f"📊 [BM25만]    Hit@1 {bm25_result['hit1']:.0%}  |  Hit@3 {bm25_result['hit3']:.0%}  |  "
      f"검색 시간 {bm25_result['time']:.0f}ms")
print(f"📊 [기본 RAG]  Hit@1 {base_result['hit1']:.0%}  |  Hit@3 {base_result['hit3']:.0%}  |  "
      f"검색 시간 {base_result['time']:.0f}ms")

ensemble_result = evaluate_retriever(ensemble_retriever, "Ensemble")

print(f"\n  기본 RAG 대비  Hit@1 {base_result['hit1']:.0%} → {ensemble_result['hit1']:.0%}"
      f"  |  Hit@3 {base_result['hit3']:.0%} → {ensemble_result['hit3']:.0%}")
print("\n💡 BM25 단독은 기본 RAG보다 낮습니다. 그런데도 합치면 올라가는 이유는")
print("   둘이 못 찾는 질문이 서로 다르기 때문입니다.")
print("   의미 검색은 'GRPO' 같은 약어에 약하고, BM25는 단어가 다르면 못 찾습니다.")

---

## 7️⃣ Parent Document — 작게 찾고 크게 돌려주기

```
문서 → 큰 조각(부모) → 작은 조각(자식)

찾을 때  : 작은 조각 (짧아서 벡터가 선명함)
돌려줄 때 : 큰 조각  (LLM한테 넉넉한 맥락을 줌)
```

### 이득이 두 군데로 나뉩니다

**하나는 점수에 잡힙니다.** 찾을 때 쓰는 조각이 100자로 짧아 벡터가 선명해지므로
검색 자체가 좋아집니다 — 이 시험지에서 Hit@1이 60% → 77%로 올랐습니다.

**하나는 점수에 안 잡힙니다.** 돌려주는 조각이 커져 LLM이 받는 맥락이 넉넉해지는데,
Hit@1과 Hit@3은 "무엇을 찾았나"만 보지 "얼마나 넉넉히 줬나"는 안 봅니다.

> 즉 **점수판이 이 기법의 절반만 담고 있습니다.** 나머지 절반은 답변 품질로 재야 하고,
> 그게 Session 08(`08_rag_evaluation`)에서 다루는 내용입니다.


In [ ]:
# 📚 Parent Document Retriever
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

docstore = InMemoryStore()
parent_vectorstore = Chroma(
    collection_name="parent_doc_demo",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)

parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 3},
)
parent_retriever.add_documents(documents)

print("✅ Parent Document Retriever 생성 완료")
print("  📊 자식 청크 100자로 검색 → 부모 청크 400자를 반환")

# 📊 채점
parent_result = evaluate_retriever(parent_retriever, "Parent Doc")

print(f"\n  기본 RAG 대비  Hit@1 {base_result['hit1']:.0%} → {parent_result['hit1']:.0%}"
      f"  |  Hit@3 {base_result['hit3']:.0%} → {parent_result['hit3']:.0%}")
print("\n💡 찾을 때 쓰는 조각이 100자로 짧아 벡터가 선명해지므로 검색이 좋아집니다.")
print("   돌려주는 조각은 400자짜리 부모라 LLM이 받는 맥락도 넉넉해집니다.")

---

## 8️⃣ 전체 비교

같은 문서, 같은 질문 30개, 같은 정답표로 네 기법을 한 표에 놓습니다.


In [ ]:
# 📊 전체 비교 — 같은 문서, 같은 질문 30개, 같은 정답표
all_results = [r for r in [
    base_result, hyde_result, reranking_result, ensemble_result, parent_result
] if r is not None]

print("=" * 72)
print(f"Advanced RAG 전체 비교 (문서 {len(documents)}개 / 질문 {len(test_questions)}개)")
print("=" * 72)
print(f"{'방법':<20}{'Hit@1':>9}{'Hit@3':>9}{'검색 시간':>12}   Hit@3 막대")
print("-" * 72)
for r in all_results:
    bar = "█" * round(r["hit3"] * 20)
    print(f"{r['name']:<18}{r['hit1']:>9.0%}{r['hit3']:>9.0%}{r['time']:>10.0f}ms   {bar}")
print("=" * 72)
print("  Hit@1 = 1등이 정답인 비율   |   Hit@3 = 정답이 3개 안에 든 비율")

# 기본 RAG와 비교
print("\n기본 RAG와 비교")
print("-" * 72)
for r in all_results:
    if r is base_result:
        continue
    d1 = (r["hit1"] - base_result["hit1"]) * 100
    d3 = (r["hit3"] - base_result["hit3"]) * 100
    배수 = r["time"] / max(base_result["time"], 0.01)
    속도 = "비슷함" if 배수 < 1.5 else f"{배수:.0f}배 느림"
    판정 = "좋아짐" if (d1 > 0 or d3 > 0) else ("그대로" if d1 == 0 and d3 == 0 else "나빠짐")
    print(f"  {r['name']:<14} Hit@1 {d1:+3.0f}%p   Hit@3 {d3:+3.0f}%p   "
          f"속도 {속도:<9}  → {판정}")

# 질문 하나하나
print("\n질문별 1등 적중 (O = 1등이 정답)")
print("-" * 72)
print(f"{'질문':<44}" + "".join(f"{r['name'][:11]:>13}" for r in all_results))
for i, q in enumerate(test_questions):
    marks = "".join(f"{('O' if r['rows'][i]['hit1'] else '·'):>13}" for r in all_results)
    print(f"{q[:42]:<44}{marks}")

### 📌 결과 읽기

기준선 — 기본 RAG: **Hit@1 60% / Hit@3 83% / 12ms**

| 기법 | Hit@1 | Hit@3 | 검색 시간 | 판정 | 왜 그런가 |
|------|-------|-------|-----------|------|-----------|
| **Reranking** | **93%** | **97%** | 69ms (6배) | 🥇 압도적 | 1단계 후보 10개 안에는 정답이 **거의 항상** 있었다. 못 찾은 게 아니라 **순서가 틀렸던 것**이고 그걸 고쳤다 |
| **Parent Doc** | **77%** | **90%** | 11ms (그대로) | 🥈 좋아짐 | 짧은 조각(100자)으로 찾으니 벡터가 선명해져 검색이 좋아졌다. 느려지지도 않는다 |
| **Ensemble** | **70%** | 83% | 10ms (그대로) | 🥉 좋아짐 | 의미 검색이 못 찾는 약어 질문을 BM25가 메운다. **단 가중치가 0.5/0.5면 이득이 사라진다** |
| **HyDE** | 60~67% | 90~93% | **1,400ms (100배)** | ❌ 비추천 | 점수는 오르지만 **돌릴 때마다 값이 달라진다**. 무엇보다 **100배 느리다** |

### Hit@1과 Hit@3을 같이 봐야 하는 이유

두 기법이 정반대 모습을 보여줍니다.

- **Reranking**: Hit@3이 83% → 97%로 조금 오르는 동안 Hit@1이 60% → 93%로 크게 뜁니다.
  → **새로 찾아낸 게 아니라, 3등 안에 있던 걸 1등으로 끌어올린 것**입니다.
- **HyDE**: Hit@3이 83% → 90~93%로 오릅니다. 없던 문서를 찾아오긴 합니다.
  다만 **돌릴 때마다 숫자가 달라지고**, 100배 느립니다.

### 남길 다섯 가지

1. **기법보다 모델**
   임베딩 모델을 영어 것으로 바꾸면 Hit@1이 60% → 17%로 무너집니다. 어떤 기법도 못 되돌립니다.
   리랭킹도 영어 모델 53% / 다국어 모델 93%입니다.
   **기법이 효과 없어 보이면 기법 말고 모델을 먼저 의심하세요.**

2. **설정 하나로 결론이 뒤집힌다**
   Ensemble은 가중치가 같으면 동점이 생기고, 동점은 먼저 적은 검색기가 전부 가져갑니다.
   리스트 순서만 바꿔도 Hit@1이 60%와 70%로 갈립니다.

3. **채점을 잘못하면 개선이 사라진다**
   검색기가 최대로 만들려는 값을 그대로 점수로 쓰면 기본 RAG가 무조건 1등입니다.
   정답표 만드는 30분이 판단 전체를 좌우합니다.

4. **점수와 시간을 같이 본다**
   HyDE는 Hit@3 몇 %p를 얻자고 100배 느려집니다. 같은 예산이면 Reranking이 압도적입니다.

5. **점수판이 기법을 다 담는지 확인하라**
   Parent Document가 주는 "넉넉한 맥락"은 Hit@1에도 Hit@3에도 나타나지 않습니다.

최종 판단은 결국 **답변이 맞았는가**로 해야 하고, 그건 Session 08(`08_rag_evaluation`)에서 다룹니다.


In [ ]:
# 💡 기법별 적합한 사용 상황 정리
print("💡 Advanced RAG 기법 선택 가이드")
print("=" * 72)

guide = [
    {
        "method": "Reranking",
        "best_for": "1차 검색이 정답을 top-N에는 넣는데 순서가 틀릴 때",
        "cost": "Cross-encoder 추론 (후보 수에 비례)",
        "caution": "반드시 다국어/한국어 리랭커를 쓸 것. 영어 모델은 역효과",
        "verdict": "가장 확실한 개선. 먼저 시도할 기법",
    },
    {
        "method": "Ensemble",
        "best_for": "약어·제품명·코드처럼 단어가 그대로 맞아야 하는 질문이 섞여 있을 때",
        "cost": "BM25 인덱스 추가 (거의 무료)",
        "caution": "약한 검색기를 섞으면 강한 쪽이 희석됨. 가중치를 실측으로 정할 것",
        "verdict": "평균보다 '실패 유형 보완'을 보고 판단",
    },
    {
        "method": "Parent Document",
        "best_for": "답변에 넓은 문맥이 필요한 긴 문서 (계약서, 논문, 매뉴얼)",
        "cost": "부모 청크 저장 공간",
        "caution": "검색 점수보다 답변 품질 쪽에서 이득이 크다",
        "verdict": "검색 지표가 아니라 생성 품질로 판단할 기법",
    },
    {
        "method": "HyDE",
        "best_for": "LLM이 이미 잘 아는 일반 도메인 + 질문이 짧고 추상적일 때",
        "cost": "질의마다 LLM 호출 1회 (가장 비싸고 느림)",
        "caution": "전문/사내 용어 코퍼스에서는 LLM이 엉뚱한 분야를 지어내 역효과",
        "verdict": "도메인 적합성을 먼저 확인하고 도입",
    },
]

for g in guide:
    print(f"\n🔹 {g['method']}")
    print(f"   적합: {g['best_for']}")
    print(f"   비용: {g['cost']}")
    print(f"   주의: {g['caution']}")
    print(f"   판정: {g['verdict']}")

print(f"\n📌 실전 도입 순서")
print(f"  1. 임베딩 모델이 해당 언어를 지원하는지 확인 (여기서 대부분의 문제가 끝난다)")
print(f"  2. 질문 20~50개에 정답 문서를 표시해 기준 점수를 잰다")
print(f"  3. Reranking부터 적용 — 비용 대비 효과가 가장 크다")
print(f"  4. 실패한 질문을 유형별로 분류해 필요한 기법을 추가")
print(f"  5. 최종 판단은 답변 품질(RAGAS 등)로 (Session 08)")

In [ ]:
# 📌 실습 정리
print("📌 오늘의 핵심 정리")
print("=" * 60)
print("  1️⃣ 평가가 먼저다")
print("     · 검색기가 최대로 만들려는 값(코사인 유사도)을 그대로 점수로 쓰면")
print("       기본 RAG가 무조건 1등이 된다 → 모든 기법이 나빠 보인다")
print("     · 사람이 만든 정답표로 Hit@1 / Hit@3 을 재야 개선이 보인다")
print()
print("  2️⃣ 기법보다 모델이 먼저다")
print("     · 임베딩: 영어 모델 Hit@1 17% → 한국어 모델 60%")
print("     · 리랭킹: 같은 기법인데 영어 모델 53%, 다국어 모델 93%")
print()
print("  3️⃣ 기법별 실측 결과  (기본 RAG = Hit@1 60% / Hit@3 83% / 12ms)")
print("     · Reranking      : Hit@1 93%  Hit@3 97%    69ms  → 압도적")
print("     · Parent Document: Hit@1 77%  Hit@3 90%    11ms  → 좋아짐, 거의 공짜")
print("     · Ensemble       : Hit@1 70%  Hit@3 83%    10ms  → 좋아짐, 거의 공짜")
print("     · HyDE           : Hit@1 60%  Hit@3 90%  1400ms  → 100배 느림, 비추천")
print()
print("  4️⃣ 설정 하나가 결론을 뒤집는다")
print("     · Ensemble 가중치가 같으면 RRF 동점이 생기고,")
print("       동점은 retrievers 리스트에 먼저 적은 쪽이 전부 가져간다")
print("     · 리스트 순서만 바꿔도 Hit@1이 60% ↔ 70% 로 달라진다")
print()
print("  5️⃣ 검색 점수가 전부는 아니다")
print("     · Parent Document가 주는 '넉넉한 맥락'은 Hit@k에 안 나타난다")
print()
print("  6️⃣ 비용을 함께 본다 — HyDE는 이득 +5~8%에 속도는 100배 느려짐")
print()
print("  7️⃣ 최종 판단은 답변 품질로 → Session 08 (RAGAS)")
print("=" * 60)

---

## 9️⃣ Streamlit으로 "비교기" 만들기 — **app2**

지금까지는 노트북에서 숫자로 비교했습니다. 이번엔 **웹 앱**으로 만들어
아무 질문이나 던지고 **여러 방법이 각각 무엇을 가져오는지 눈으로** 봅니다.

### 무엇을 보여주는 앱인가

```
                 질문 하나
                     │
    ┌────────┬───────┼────────┬─────────┬────────┐
  기본 RAG   BM25  Ensemble Reranking ParentDoc  HyDE
    │        │       │        │         │        │
    └────────┴───────┴────────┴─────────┴────────┘
                     │
       각각의 검색 시간 · 가져온 문서 · 서로 몇 개나 겹치는지
```

- 왼쪽에서 **비교할 방법을 골라** 체크하면 그만큼 열이 생깁니다
- **검색 시간**이 방법별로 함께 표시됩니다 (기준 대비 몇 배 느린지)
- 아래에 **서로 몇 개나 같은 문서를 가져왔는지**가 나옵니다 — 겹침이 적을수록 서로 다른 걸 본다는 뜻
- HyDE를 켜면 **LLM이 지어낸 가짜 문서**를 직접 펼쳐볼 수 있습니다
- 원하면 **답변까지 만들어** 방법별로 나란히 비교합니다

### 준비물

- **API 키 없어도 됩니다.** 검색 방법 5개는 그대로 동작하고, HyDE와 답변 생성만 빠집니다.
- 문서는 **샘플 14개가 내장**되어 있어 바로 시작할 수 있고, **내 PDF를 올려도** 됩니다.
- 노트북과 **똑같은 모델**을 씁니다 (KoSimCSE 임베딩 + bge-reranker-v2-m3).
  이 노트북을 이미 돌렸다면 모델이 캐시에 있어 바로 뜹니다.

### 만드는 순서

| 셀 | 하는 일 |
|----|---------|
| ⓪ | 폴더 `app2/` 만들기 |
| ① | `app.py` 쓰기 |
| ② | `requirements.txt` 쓰기 |
| ③ | `.env.example` 쓰기 |
| ④ | 실행 명령 확인 → 터미널에서 실행 |

> ⚠️ `%%writefile` 은 **폴더를 자동으로 만들지 않습니다.** ⓪을 건너뛰면
> `FileNotFoundError: app2/app.py` 가 납니다.

> 💡 Session 04에서 만든 `streamlit_app/` 과 겹치지 않도록 **다른 폴더**에 만듭니다.
> 04는 "PDF 질의응답 챗봇", 05는 "검색 방법 비교기"로 서로 다른 앱입니다.

> 💡 Streamlit 앱은 노트북 셀에서 실행할 수 없습니다 (별도 프로세스입니다).
> 노트북은 **파일을 만드는 용도**, 실행은 터미널에서 합니다.


### ⓪ 폴더 만들기

In [ ]:
# ⓪ 앱을 담을 폴더 만들기
# %%writefile 은 파일만 쓰고 폴더는 만들어주지 않는다. 반드시 먼저 실행할 것.
import os
from pathlib import Path

APP_DIR = Path("app2")     # Session 04는 app1/, 이 세션은 app2/
APP_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 폴더 준비 완료: {APP_DIR.resolve()}")
print(f"   현재 내용: {sorted(p.name for p in APP_DIR.iterdir()) or '(비어 있음)'}")

### ① `app.py` 쓰기

앱의 본체입니다. 길지만 구조는 단순합니다.

1. **모델 로딩** — `@st.cache_resource` 로 앱 실행 중 한 번만 로딩
2. **인덱스 구축** — 업로드한 PDF(또는 샘플)를 조각내어 벡터스토어·BM25·Parent 준비
3. **검색 방법 6개** — 각각 `(인덱스, 질문, 개수) → 문서 목록` 함수 하나로 통일
4. **화면** — 고른 방법 수만큼 열을 만들어 나란히 출력

> 무거운 모델은 **시간 측정 전에 미리 로딩**합니다.
> 안 그러면 첫 검색에 모델 로딩 시간이 섞여 Reranking이 수천 ms로 보입니다.


In [ ]:
%%writefile app2/app.py
"""
Advanced RAG 검색 방법 비교기 (Streamlit) — Session 05 실습
==========================================================
같은 질문을 여러 검색 방법에 동시에 던져 결과를 나란히 비교합니다.

사용법:
    1) cd streamlit_app
    2) pip install -r requirements.txt
    3) streamlit run app.py

비교 대상:
    - 기본 RAG      : 의미(벡터) 검색만
    - BM25          : 키워드 검색만
    - Ensemble      : 위 둘을 RRF로 합침
    - Reranking     : 넓게 찾고 Cross-encoder로 순서 다시 매김
    - Parent Doc    : 작은 조각으로 찾고 큰 조각을 돌려줌
    - HyDE          : LLM이 가짜 답변을 만들고 그걸로 검색 (API 키 필요)

API 키가 없어도 HyDE와 답변 생성만 빠지고 나머지는 모두 동작합니다.
"""
from __future__ import annotations

import os
import tempfile
import time
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv

from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Chroma
from langchain.retrievers import EnsembleRetriever, ParentDocumentRetriever
from langchain.storage import InMemoryStore

load_dotenv()

# === 설정 =========================================================
EMBEDDING_MODEL = "BM-K/KoSimCSE-roberta-multitask"   # 노트북과 동일
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"            # 노트북과 동일

st.set_page_config(page_title="Advanced RAG 비교기", page_icon="🔍", layout="wide")


# === 샘플 문서 (PDF 없이 바로 써보기) ===============================
SAMPLE_DOCS = [
    ("소형 언어 모델", """소형 언어 모델(sLLM)은 1B에서 7B 규모의 경량 언어 모델입니다.
Phi-3, Gemma, Qwen2.5 등이 대표적이며 온디바이스 추론과 저비용 서빙에 적합합니다.
큰 모델 대비 성능 손실이 크지 않아 특정 도메인에서는 충분한 대안이 됩니다."""),
    ("PEFT", """LoRA는 저랭크 행렬 분해로 학습 파라미터 수를 줄이는 PEFT 기법입니다.
QLoRA는 4bit 양자화와 LoRA를 결합해 RTX 4060 같은 소비자급 GPU에서도 7B 모델 학습을 가능하게 합니다.
어댑터, 프리픽스 튜닝, IA3 등도 대표적인 PEFT 계열입니다."""),
    ("양자화", """양자화는 모델 가중치를 저정밀도로 바꿔 메모리를 줄이는 기법입니다.
GPTQ와 AWQ는 대표적인 사후 양자화 방식이며, GGUF는 CPU 추론에 널리 쓰이는 파일 포맷입니다.
비트를 낮출수록 메모리는 줄지만 정확도 손실이 생길 수 있습니다."""),
    ("벡터 DB", """벡터 데이터베이스는 고차원 벡터를 저장하고 유사도 검색을 수행합니다.
ChromaDB는 오픈소스 임베디드 벡터 DB이고, FAISS는 Meta가 만든 고속 검색 라이브러리입니다.
Weaviate는 GraphQL 기반 검색 엔진이며, Pinecone은 관리형 클라우드 서비스입니다."""),
    ("ANN 인덱스", """HNSW는 계층 그래프를 타고 이웃을 탐색하는 근사 최근접 이웃 알고리즘입니다.
IVF는 벡터를 클러스터로 나눈 뒤 가까운 클러스터만 탐색하며, nprobe로 정확도를 조절합니다.
PQ는 벡터를 압축해 메모리를 절감하지만 정확도를 일부 희생합니다."""),
    ("정렬", """RLHF는 사람의 선호를 보상 모델로 학습해 정책에 반영하는 정렬 기법입니다.
DPO는 보상 모델 없이 선호 쌍으로 직접 최적화합니다.
이런 정렬 기법은 모델의 안전성과 유용성을 높이는 데 사용됩니다."""),
    ("강화학습", """PPO는 정책이 한 번에 크게 변하지 않도록 제한해 안정적인 업데이트를 보장합니다.
GRPO는 DeepSeek이 제안한 효율적인 정책 최적화 방법입니다.
GRPO는 가치 함수 없이 그룹 내 상대 비교로 이점을 추정해 메모리를 아낍니다."""),
    ("프롬프트", """Chain-of-Thought는 모델이 단계별 추론 과정을 서술하도록 유도하는 기법입니다.
복잡한 수리·논리 문제에서 정답률을 크게 끌어올립니다.
Self-Consistency는 여러 추론 경로를 뽑아 다수결로 답을 정합니다."""),
    ("어텐션", """셀프 어텐션은 Query, Key, Value 세 행렬의 곱으로 토큰 간 관계를 계산합니다.
멀티헤드 어텐션은 여러 부분공간에서 병렬로 어텐션을 수행합니다.
Flash Attention은 메모리 접근을 최적화해 긴 시퀀스 학습 속도를 크게 높입니다.
그룹 쿼리 어텐션(GQA)은 Key와 Value 헤드를 공유해 추론 메모리를 줄입니다."""),
    ("RAG 기본", """RAG는 외부 지식을 검색해 LLM 답변에 근거를 제공하는 기술입니다.
문서 로딩, 청킹, 임베딩, 검색, 생성의 다섯 단계로 구성됩니다.
파인튜닝 없이 최신 정보를 반영할 수 있다는 것이 장점입니다.
검색이 실패하면 아무리 좋은 LLM도 옳은 답을 낼 수 없습니다."""),
    ("하이브리드 검색", """하이브리드 검색은 BM25 같은 키워드 검색과 벡터 검색을 결합합니다.
고유명사나 코드처럼 정확한 토큰 일치가 중요한 질의에서는 키워드 검색이 강합니다.
RRF는 두 순위 목록을 상호 순위 역수로 융합하는 방법입니다."""),
    ("임베딩", """임베딩은 텍스트를 고차원 벡터로 바꿔 의미를 수치화하는 과정입니다.
의미가 비슷한 문장은 벡터 공간에서 가까이 놓입니다.
한국어에는 KoSimCSE, multilingual-e5 같은 한국어 지원 모델이 적합합니다.
영어 전용 모델을 한국어에 쓰면 검색 품질이 급격히 무너집니다."""),
    ("RAG 평가", """RAGAS는 RAG 파이프라인을 지표화해 평가하는 프레임워크입니다.
Faithfulness는 답변이 근거 문서에 충실한지를 봅니다.
검색 품질은 정답 문서를 얼마나 잘 올렸는지로 따로 재야 합니다."""),
    ("에이전트", """LLM 에이전트는 도구를 호출하며 여러 단계로 문제를 해결하는 시스템입니다.
ReAct는 추론과 행동을 번갈아 수행하는 대표적인 패턴입니다.
MCP는 모델과 외부 도구를 연결하는 표준 프로토콜입니다."""),
]


# === 모델 로딩 (앱 실행 중 한 번만) ==================================
@st.cache_resource(show_spinner="임베딩 모델 로딩 중... (최초 1회)")
def load_embeddings():
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        encode_kwargs={"normalize_embeddings": True},
    )


@st.cache_resource(show_spinner="Reranker 로딩 중... (최초 1회, 약 2.2GB)")
def load_reranker():
    from sentence_transformers import CrossEncoder
    return CrossEncoder(RERANKER_MODEL, max_length=512)


# === 인덱스 구축 ===================================================
@st.cache_resource(show_spinner="문서 인덱싱 중...")
def build_index(file_key, chunk_size: int, chunk_overlap: int, _docs: list[Document]):
    """검색에 필요한 것들을 한 번에 만든다. file_key는 캐시 키 역할."""
    embeddings = load_embeddings()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    splits = splitter.split_documents(_docs)

    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        collection_name=f"cmp_{abs(hash(file_key)) % 10**8}",
        collection_metadata={"hnsw:space": "cosine"},
    )

    bm25 = BM25Retriever.from_documents(splits)

    # Parent Document — 작은 조각으로 찾고 큰 조각을 돌려준다
    parent_vs = Chroma(
        collection_name=f"par_{abs(hash(file_key)) % 10**8}",
        embedding_function=embeddings,
        collection_metadata={"hnsw:space": "cosine"},
    )
    parent = ParentDocumentRetriever(
        vectorstore=parent_vs,
        docstore=InMemoryStore(),
        child_splitter=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20),
        parent_splitter=RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50),
    )
    parent.add_documents(_docs)

    return {"splits": splits, "vectorstore": vectorstore,
            "bm25": bm25, "parent": parent}


# === 검색 방법들 ===================================================
def search_basic(idx, query, k):
    return idx["vectorstore"].similarity_search(query, k=k)


def search_bm25(idx, query, k):
    idx["bm25"].k = k
    return idx["bm25"].invoke(query)


def search_ensemble(idx, query, k):
    idx["bm25"].k = k
    ens = EnsembleRetriever(
        retrievers=[idx["bm25"], idx["vectorstore"].as_retriever(search_kwargs={"k": k})],
        weights=[0.4, 0.6],      # 0.5/0.5로 두면 동점이 생겨 순서에 좌우된다
    )
    return ens.invoke(query)[:k]


def search_rerank(idx, query, k, fetch_k=10):
    candidates = idx["vectorstore"].similarity_search(query, k=fetch_k)
    if not candidates:
        return []
    scores = load_reranker().predict([[query, d.page_content] for d in candidates])
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:k]]


def search_parent(idx, query, k):
    idx["parent"].search_kwargs = {"k": k}
    return idx["parent"].invoke(query)[:k]


def make_hyde_search(llm):
    def search_hyde(idx, query, k):
        prompt = (f"다음 질문에 대한 답변을 기술 문서의 한 문단처럼 3문장 이내로 작성하세요.\n"
                  f"사실 여부는 중요하지 않습니다.\n\n질문: {query}\n\n답변 문단:")
        fake = llm.invoke(prompt)
        fake = fake if isinstance(fake, str) else fake.content
        st.session_state["last_hyde"] = fake.strip()
        return idx["vectorstore"].similarity_search(fake, k=k)
    return search_hyde


# === 사이드바 ======================================================
with st.sidebar:
    st.header("⚙️ 설정")

    st.subheader("📄 문서")
    source = st.radio("무엇으로 검색할까요?",
                      ["샘플 문서 8개 (바로 시작)", "내 PDF 올리기"],
                      label_visibility="collapsed")

    uploaded = None
    if source == "내 PDF 올리기":
        uploaded = st.file_uploader("PDF 선택", type=["pdf"], accept_multiple_files=True)

    st.divider()
    st.subheader("✂️ 조각내기")
    chunk_size = st.slider("조각 크기 (자)", 100, 1000, 200, 50)
    chunk_overlap = st.slider("겹치는 부분 (자)", 0, 200, 30, 10)
    top_k = st.slider("가져올 개수", 1, 5, 3)

    st.divider()
    st.subheader("🔍 비교할 방법")
    api_key = os.getenv("OPENAI_API_KEY")
    has_key = bool(api_key) and not api_key.startswith("sk-your")

    picks = {
        "기본 RAG": st.checkbox("기본 RAG (의미 검색)", value=True),
        "BM25": st.checkbox("BM25 (키워드 검색)", value=False),
        "Ensemble": st.checkbox("Ensemble (BM25+의미)", value=True),
        "Reranking": st.checkbox("Reranking (2단계)", value=True),
        "Parent Doc": st.checkbox("Parent Document", value=False),
        "HyDE": st.checkbox("HyDE (API 키 필요)", value=False, disabled=not has_key),
    }

    st.divider()
    st.caption(f"임베딩: `{EMBEDDING_MODEL}`")
    st.caption(f"Reranker: `{RERANKER_MODEL}`")
    if has_key:
        st.success("OpenAI API 키 확인됨")
    else:
        st.info("API 키 없음 — HyDE와 답변 생성은 비활성화됩니다")


# === 본문 ==========================================================
st.title("🔍 Advanced RAG 검색 방법 비교기")
st.caption("같은 질문을 여러 검색 방법에 동시에 던져, 무엇을 가져오는지 나란히 봅니다.")

# 문서 준비
if source == "내 PDF 올리기":
    if not uploaded:
        st.info("👈 왼쪽에서 PDF를 올리거나 '샘플 문서'를 선택하세요.")
        st.stop()
    docs, key_parts = [], []
    for f in uploaded:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(f.getvalue())
            tmp_path = tmp.name
        try:
            pages = PyPDFLoader(tmp_path).load()
            for p in pages:
                p.metadata["source"] = f.name
            docs.extend(pages)
        finally:
            os.unlink(tmp_path)
        key_parts.append(f"{f.name}:{len(f.getvalue())}")
    file_key = "|".join(key_parts)
else:
    docs = [Document(page_content=body, metadata={"source": f"{name}.txt"})
            for name, body in SAMPLE_DOCS]
    file_key = "sample-v2"

idx = build_index(file_key, chunk_size, chunk_overlap, docs)
st.success(f"문서 {len(docs)}개 → 조각 {len(idx['splits'])}개 준비 완료")

# 검색 방법 모으기
llm = None
if has_key:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

METHODS = {
    "기본 RAG": search_basic,
    "BM25": search_bm25,
    "Ensemble": search_ensemble,
    "Reranking": search_rerank,
    "Parent Doc": search_parent,
}
if llm is not None:
    METHODS["HyDE"] = make_hyde_search(llm)

selected = [name for name, on in picks.items() if on and name in METHODS]

# 질문 입력
examples = ["GRPO가 뭐야?", "RTX 4060으로도 학습이 되나?",
            "CPU에서 모델 돌릴 때 쓰는 파일 포맷은?", "단계별로 생각하게 시키면 정답률이 오르나?"]
st.write("**예시 질문** (누르면 입력됩니다)")
cols = st.columns(len(examples))
for col, ex in zip(cols, examples):
    if col.button(ex, use_container_width=True):
        st.session_state["query"] = ex

query = st.text_input("질문", key="query", placeholder="문서에 대해 물어보세요…")

if not query:
    st.stop()
if not selected:
    st.warning("👈 비교할 방법을 하나 이상 선택하세요.")
    st.stop()

# 무거운 모델은 측정 전에 미리 로딩한다.
# (로딩 시간이 검색 시간에 섞이면 Reranking이 수천 ms로 보인다)
if "Reranking" in selected:
    load_reranker()

# 검색 실행
results, timings = {}, {}
for name in selected:
    start = time.time()
    try:
        results[name] = METHODS[name](idx, query, top_k)
        timings[name] = (time.time() - start) * 1000
    except Exception as e:
        results[name] = []
        timings[name] = 0
        st.error(f"{name} 실패: {e}")

# 속도 요약
st.subheader("⏱️ 검색 시간")
tcols = st.columns(len(selected))
fastest = min(timings.values()) if timings else 1
for col, name in zip(tcols, selected):
    ms = timings[name]
    ratio = ms / fastest if fastest else 1
    col.metric(name, f"{ms:.0f}ms",
               delta=None if ratio < 1.5 else f"{ratio:.0f}배 느림",
               delta_color="inverse")

if "HyDE" in selected and st.session_state.get("last_hyde"):
    with st.expander("🔮 HyDE가 지어낸 가짜 문서 보기"):
        st.write(st.session_state["last_hyde"])
        st.caption("이 문서로 검색합니다. 엉뚱한 분야를 지어내면 검색도 함께 틀어집니다.")

# 결과 나란히
st.subheader("📄 가져온 문서")
rcols = st.columns(len(selected))
for col, name in zip(rcols, selected):
    with col:
        st.markdown(f"### {name}")
        docs_found = results[name]
        if not docs_found:
            st.write("결과 없음")
            continue
        for rank, d in enumerate(docs_found, 1):
            src = d.metadata.get("source", "?")
            st.markdown(f"**{rank}위** · `{src}`")
            st.caption(d.page_content[:200].replace("\n", " ") +
                       ("..." if len(d.page_content) > 200 else ""))

# 몇 개가 겹치는지
if len(selected) > 1:
    st.subheader("🔗 방법끼리 얼마나 같은 문서를 가져왔나")
    base_name = selected[0]
    base_set = {d.page_content for d in results[base_name]}
    lines = []
    for name in selected[1:]:
        overlap = len(base_set & {d.page_content for d in results[name]})
        lines.append(f"- **{base_name}** vs **{name}** — {top_k}개 중 **{overlap}개** 같음")
    st.markdown("\n".join(lines))
    st.caption("겹치는 게 적을수록 두 방법이 서로 다른 문서를 본다는 뜻입니다.")

# 답변까지 비교 (선택)
if llm is not None and st.checkbox("이 문서들로 답변까지 만들어 비교하기"):
    st.subheader("💬 방법별 답변")
    acols = st.columns(len(selected))
    for col, name in zip(acols, selected):
        with col:
            st.markdown(f"### {name}")
            context = "\n\n".join(d.page_content for d in results[name])
            if not context:
                st.write("검색 결과가 없어 답할 수 없습니다.")
                continue
            prompt = (f"아래 문서만 참고해 한국어로 간단히 답하세요.\n"
                      f"문서에 없으면 '문서에서 찾을 수 없습니다'라고 답하세요.\n\n"
                      f"문서:\n{context}\n\n질문: {query}\n답변:")
            with st.spinner("생성 중..."):
                out = llm.invoke(prompt)
                st.write(out if isinstance(out, str) else out.content)

### ② `requirements.txt` 쓰기

In [ ]:
%%writefile app2/requirements.txt
streamlit>=1.30
langchain>=0.3
langchain-community>=0.3
langchain-openai>=0.2
chromadb>=0.5
sentence-transformers>=3.0
rank-bm25>=0.2
pypdf>=4.0
python-dotenv>=1.0

### ③ `.env.example` 쓰기

HyDE와 답변 생성에만 필요합니다. **없어도 앱은 돌아갑니다.**


In [ ]:
%%writefile app2/.env.example
# HyDE 와 '답변까지 비교' 기능에만 필요합니다.
# 키가 없어도 나머지 검색 방법 5개는 모두 동작합니다.
OPENAI_API_KEY=sk-your-key-here

### ④ 실행하기

In [ ]:
# 실행 방법 (복사해서 터미널에서 사용)
print(r"""
# 1) 폴더 이동
cd app2

# 2) 패키지 설치 (이미 노트북 환경이면 건너뛰어도 됩니다)
pip install -r requirements.txt

# 3) (선택) API 키 — HyDE 와 답변 생성에만 필요
cp .env.example .env      # .env 를 열어 OPENAI_API_KEY 입력

# 4) 실행
streamlit run app.py

# → 브라우저가 열립니다  http://localhost:8501
#   원격 서버라면:  streamlit run app.py --server.address 0.0.0.0
""")

print("처음 실행할 때는 모델을 내려받느라 몇 분 걸립니다 (임베딩 424MB + Reranker 2.2GB).")
print("이 노트북을 이미 돌렸다면 캐시에 있으므로 바로 뜹니다.")

---

## 🎯 실습 과제

1️⃣ **임베딩 모델 되돌리기** — `EMBEDDING_MODEL`을 `all-MiniLM-L6-v2`로 바꿔 전체를 다시 돌리세요.
   기본 RAG와 Advanced 기법 중 어느 쪽이 더 크게 무너지나요?

2️⃣ **리랭킹 모델 바꾸기** — `RERANKER_MODEL`을 `cross-encoder/ms-marco-MiniLM-L-6-v2`로 바꾸세요.
   같은 "리랭킹"인데 왜 결과가 달라지나요?

3️⃣ **앙상블 가중치** — `weights`를 `[0.5, 0.5]`, `[0.3, 0.7]`, `[0.7, 0.3]`으로 바꿔보세요.
   `[0.5, 0.5]`일 때만 `retrievers` 순서에 결과가 좌우되는 이유를 설명해보세요.

4️⃣ **잘못된 채점 재현** — 점수를 "질문과 검색 문서의 코사인 유사도 평균"으로 바꿔 다시 돌리세요.
   모든 기법이 기본 RAG보다 나빠지는 걸 확인하고, 왜 그런지 설명해보세요.

5️⃣ **내 문서로** — 문서 20개와 질문 10개를 만들고 정답을 직접 표시해 같은 시험을 돌려보세요.
   도메인이 바뀌면 승자도 바뀝니다.

---

## 📚 참고 자료
- [HyDE 논문](https://arxiv.org/abs/2212.10496)
- [BGE Reranker](https://huggingface.co/BAAI/bge-reranker-v2-m3)
- [Sentence Transformers Cross-Encoders](https://www.sbert.net/docs/cross_encoder/usage/usage.html)
- [LangChain Ensemble Retriever](https://python.langchain.com/docs/how_to/ensemble_retriever/)
- [LangChain Parent Document Retriever](https://python.langchain.com/docs/how_to/parent_document_retriever/)
- [RRF 논문](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [ANN-Benchmarks](https://ann-benchmarks.com/)
